# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion  
**Query Expansion Method:** HyDE (Hypothetical Document Embedding) using Gemini  

---

## Pipeline Overview

```
User Query
    │
    ▼
Part 4: HyDE Query Expansion (Gemini generates a hypothetical answer)
    │
    ▼
Part 2: Hybrid Retrieval (BM25 + SBERT fused with RRF)
    │
    ▼
Part 3: Cross-Encoder Re-Ranking
    │
    ▼
Part 5: LLM Generation (Gemini)
    │
    ▼
Final Answer
```

## Cell 1 — Install Dependencies

In [1]:
# Install all required packages
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-google-genai langchain-core numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.5 MB/s eta 0:00:00


## Cell 2 — API Key Setup

In [2]:
import os
import getpass

# Enter your Google API key (Gemini)
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key (Gemini): ")

print(f"Google API key loaded: {'YES' if os.getenv('GOOGLE_API_KEY') else 'NO — please re-enter above'}")

Enter your Google API Key (Gemini): ··········
Google API key loaded: YES


---
## Part 1 — Document Corpus Setup

We build a corpus of **12 documents** on AI/ML topics. Design decisions:
- Documents 0–2 cover **neural network training** from three different angles (backpropagation, optimizers, regularisation)
- Documents 3–5 cover **attention & transformers** from different angles
- Document 6 uses **BM25-friendly proper noun jargon** (`cross-entropy`, `Softmax`) that a sparse retriever excels at
- Documents 7–11 provide coverage of retrieval, fine-tuning, quantization, and evaluation topics

In [3]:
# ============================================================
# PART 1 — Document Corpus (12 documents)
# ============================================================

corpus = [
    # ---- Neural Network Training (3 related but distinct sub-topics) ----
    # doc_0
    "Backpropagation computes gradients by applying the chain rule layer by layer, "
    "allowing the network to attribute error to each weight and update them accordingly.",

    # doc_1
    "Stochastic gradient descent and its variants — Adam, RMSProp, and AdaGrad — "
    "are optimization algorithms that iteratively adjust model weights to minimize the loss function.",

    # doc_2
    "Regularization techniques such as dropout, weight decay, and batch normalization "
    "prevent overfitting by constraining the model's capacity during neural network training.",

    # ---- Attention & Transformers (3 related but distinct sub-topics) ----
    # doc_3
    "Self-attention allows every token in a sequence to attend to every other token, "
    "capturing long-range dependencies that recurrent networks struggle with.",

    # doc_4
    "Multi-head attention runs several attention operations in parallel, each learning "
    "to focus on different syntactic and semantic relationships within the input.",

    # doc_5
    "Positional encodings are added to token embeddings in transformers to inject information "
    "about token order, since self-attention is permutation-invariant by design.",

    # ---- Keyword / jargon-heavy document (BM25 advantage) ----
    # doc_6  <-- BM25 will find this easily via exact-match on 'cross-entropy', 'softmax', 'logits'
    "The cross-entropy loss function measures the divergence between predicted Softmax probabilities "
    "and one-hot target labels; minimizing cross-entropy over logits is equivalent to maximum likelihood estimation.",

    # ---- Retrieval & RAG ----
    # doc_7
    "Retrieval-Augmented Generation (RAG) grounds LLM outputs by retrieving relevant documents "
    "at inference time, reducing hallucinations and enabling access to up-to-date knowledge.",

    # doc_8
    "BM25 is a sparse retrieval algorithm that ranks documents using term frequency and "
    "inverse document frequency with saturation and length normalisation.",

    # ---- Fine-Tuning ----
    # doc_9
    "LoRA (Low-Rank Adaptation) fine-tunes large language models by injecting trainable "
    "low-rank matrices into attention layers while keeping the base model weights frozen.",

    # ---- Quantization ----
    # doc_10
    "Quantization compresses model weights from FP32 to INT8 or INT4, reducing memory "
    "footprint and inference latency with only a small degradation in accuracy.",

    # ---- Evaluation ----
    # doc_11
    "BLEU and ROUGE are n-gram overlap metrics used to evaluate the quality of generated "
    "text against reference outputs in tasks like summarization and machine translation.",
]

print(f"Corpus loaded: {len(corpus)} documents")
print("\nCorpus preview:")
for i, doc in enumerate(corpus):
    print(f"  doc_{i:02d}: {doc[:80]}...")

Corpus loaded: 12 documents

Corpus preview:
  doc_00: Backpropagation computes gradients by applying the chain rule layer by layer, al...
  doc_01: Stochastic gradient descent and its variants — Adam, RMSProp, and AdaGrad — are ...
  doc_02: Regularization techniques such as dropout, weight decay, and batch normalization...
  doc_03: Self-attention allows every token in a sequence to attend to every other token, ...
  doc_04: Multi-head attention runs several attention operations in parallel, each learnin...
  doc_05: Positional encodings are added to token embeddings in transformers to inject inf...
  doc_06: The cross-entropy loss function measures the divergence between predicted Softma...
  doc_07: Retrieval-Augmented Generation (RAG) grounds LLM outputs by retrieving relevant ...
  doc_08: BM25 is a sparse retrieval algorithm that ranks documents using term frequency a...
  doc_09: LoRA (Low-Rank Adaptation) fine-tunes large language models by injecting trainab...
  doc_10: Quant

---
## Part 2 — Hybrid Retrieval (BM25 + SBERT + RRF)

**Reciprocal Rank Fusion (RRF)** fuses two ranked lists in a scale-agnostic way:

$$\text{RRF}(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)}$$

where $k=60$ dampens the advantage of rank-1 documents.

In [4]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class HybridRetriever:
    """
    Hybrid retriever combining BM25 (sparse) and SBERT (dense)
    fused via Reciprocal Rank Fusion (RRF).

    Parameters
    ----------
    corpus : list[str]  – list of document strings
    k      : int        – RRF smoothing constant (default 60)
    """

    def __init__(self, corpus: list, k: int = 60):
        self.corpus = corpus
        self.k = k

        # ── BM25 index (sparse) ──
        # Lowercase + whitespace tokenization as required by BM25
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # ── SBERT index (dense) ──
        print("Loading SBERT model...")
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        # Pre-encode all corpus documents (L2-normalized for cosine via dot product)
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        norms = np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True)
        self.doc_embeddings = self.doc_embeddings / norms  # unit-normalize
        print(f"HybridRetriever ready — {len(corpus)} docs indexed.")

    # ------------------------------------------------------------------
    def _bm25_ranked_list(self, query: str) -> list:
        """Return list of (doc_id, bm25_score) sorted by descending score."""
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        ranked = np.argsort(scores)[::-1]  # highest score first
        return [(int(idx), float(scores[idx])) for idx in ranked]

    def _sbert_ranked_list(self, query: str) -> list:
        """Return list of (doc_id, cosine_score) sorted by descending score."""
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)  # unit-normalize
        scores = self.doc_embeddings @ q_vec    # dot-product = cosine (both normalized)
        ranked = np.argsort(scores)[::-1]
        return [(int(idx), float(scores[idx])) for idx in ranked]

    # ------------------------------------------------------------------
    def retrieve(self, query: str, top_k: int = 5) -> list:
        """
        Hybrid retrieval with RRF fusion.

        Returns
        -------
        list of dicts, each with keys:
            doc_id    – index into the corpus
            rrf_score – fused RRF score (higher = more relevant)
            bm25_rank – 1-indexed rank in the BM25 list
            sbert_rank– 1-indexed rank in the SBERT list
            text      – document text
        """
        bm25_list  = self._bm25_ranked_list(query)   # full ranking
        sbert_list = self._sbert_ranked_list(query)  # full ranking

        # Build rank-lookup dicts  {doc_id: 1-indexed rank}
        bm25_rank  = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(bm25_list)}
        sbert_rank = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(sbert_list)}

        # Compute RRF score for every document
        k = self.k
        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                1.0 / (k + bm25_rank[doc_id]) +
                1.0 / (k + sbert_rank[doc_id])
            )

        # Sort by RRF score descending and take top_k
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, rrf_score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(rrf_score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })

        return results


# ── Instantiate the retriever ──
retriever = HybridRetriever(corpus, k=60)

Loading SBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever ready — 12 docs indexed.


In [5]:
# ── Quick smoke-test of HybridRetriever ──

test_query = "how do neural networks learn?"
print(f"Query: '{test_query}'\n")
print(f"{'Rank':<5} {'doc_id':<8} {'RRF':<10} {'BM25_rank':<12} {'SBERT_rank':<12} Text")
print("-" * 100)

for rank, hit in enumerate(retriever.retrieve(test_query, top_k=5), 1):
    print(
        f"{rank:<5} {hit['doc_id']:<8} {hit['rrf_score']:<10.6f} "
        f"{hit['bm25_rank']:<12} {hit['sbert_rank']:<12} "
        f"{hit['text'][:70]}..."
    )

Query: 'how do neural networks learn?'

Rank  doc_id   RRF        BM25_rank    SBERT_rank   Text
----------------------------------------------------------------------------------------------------
1     2        0.032002   2            3            Regularization techniques such as dropout, weight decay, and batch nor...
2     3        0.031778   1            5            Self-attention allows every token in a sequence to attend to every oth...
3     9        0.030303   6            6            LoRA (Low-Rank Adaptation) fine-tunes large language models by injecti...
4     0        0.030282   12           1            Backpropagation computes gradients by applying the chain rule layer by...
5     1        0.030214   11           2            Stochastic gradient descent and its variants — Adam, RMSProp, and AdaG...


---
## Part 3 — Cross-Encoder Re-Ranker

A **cross-encoder** encodes query + document *jointly*, allowing token-level attention between them. This is far more accurate than a bi-encoder but too slow to run on the entire corpus — so we run it only on the top-K candidates from hybrid retrieval.

> **Note:** Cross-encoder scores are raw logits — they can be negative. Higher (less negative) = more relevant.

In [6]:
from sentence_transformers import CrossEncoder

# Load cross-encoder model
print("Loading cross-encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: list, top_k: int = 3) -> list:
    """
    Re-rank retrieved candidates using a cross-encoder.

    Parameters
    ----------
    query      : str        – the ORIGINAL user query (not HyDE-expanded)
    candidates : list[dict] – dicts returned by HybridRetriever.retrieve()
    top_k      : int        – number of top documents to return after re-ranking

    Returns
    -------
    list of dicts with an added 'ce_score' key, sorted by ce_score descending
    """
    if not candidates:
        return []

    # Build (query, document_text) pairs for the cross-encoder
    pairs = [[query, hit["text"]] for hit in candidates]

    # Score all pairs in one batch (can be negative — that's normal)
    ce_scores = cross_encoder.predict(pairs)

    # Attach scores to candidate dicts
    scored = []
    for hit, score in zip(candidates, ce_scores):
        entry = dict(hit)                     # shallow copy so we don't mutate the original
        entry["ce_score"] = float(score)
        scored.append(entry)

    # Sort by cross-encoder score descending (higher = more relevant)
    scored.sort(key=lambda x: x["ce_score"], reverse=True)

    return scored[:top_k]


print("rerank() function defined.")

Loading cross-encoder model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
rerank() function defined.


In [7]:
# ── Smoke-test the reranker ──

test_query = "how do neural networks learn?"
candidates = retriever.retrieve(test_query, top_k=7)
reranked   = rerank(test_query, candidates, top_k=3)

print(f"Query: '{test_query}'")
print("\nTop-3 after Cross-Encoder Re-Ranking:")
print(f"{'Rank':<5} {'CE Score':<12} {'doc_id':<8} Text")
print("-" * 90)
for rank, hit in enumerate(reranked, 1):
    print(f"{rank:<5} {hit['ce_score']:<12.4f} {hit['doc_id']:<8} {hit['text'][:70]}...")

Query: 'how do neural networks learn?'

Top-3 after Cross-Encoder Re-Ranking:
Rank  CE Score     doc_id   Text
------------------------------------------------------------------------------------------
1     -1.9340      2        Regularization techniques such as dropout, weight decay, and batch nor...
2     -7.7242      4        Multi-head attention runs several attention operations in parallel, ea...
3     -9.2720      3        Self-attention allows every token in a sequence to attend to every oth...


---
## Part 4 — Query Expansion: HyDE (Hypothetical Document Embedding)

**HyDE** (Gao et al., 2022) bridges the vocabulary gap between short user queries and detailed document language:

1. User query → Gemini generates a *hypothetical ideal answer* (written in textbook style)
2. That hypothetical answer is embedded → used as the retrieval vector

**Why it works:** The hypothetical answer uses the same technical vocabulary as real corpus documents, so the embedding lands much closer to relevant documents in vector space than a 5-word query would.

> `temperature=0.0` is used for determinism — we want a factual, stable hypothetical document, not a creative paraphrase.

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Gemini model (temperature=0 for deterministic hypothetical docs) ──
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.0
)

# ── HyDE prompt ──
hyde_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a technical writer creating content for an AI/ML textbook. "
        "Write a single factual paragraph (3–5 sentences) that would directly answer "
        "the following question. Use precise technical vocabulary. "
        "Do NOT mention the question itself — write as if it were a textbook excerpt."
    ),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()


def expand_query_hyde(user_query: str) -> str:
    """
    HyDE query expansion.
    Returns a hypothetical document string to use as the retrieval query.

    Parameters
    ----------
    user_query : str – the original short user question

    Returns
    -------
    str – a longer hypothetical answer that will embed close to relevant docs
    """
    hypothetical_doc = hyde_chain.invoke({"query": user_query})
    return hypothetical_doc.strip()


print("HyDE query expansion function defined.")

HyDE query expansion function defined.


In [11]:
# ── Demo: show HyDE in action ──

demo_query = "how do transformers encode meaning?"
print(f"Original query : '{demo_query}'")
print()

hypothetical = expand_query_hyde(demo_query)
print(f"HyDE hypothetical document:\n{hypothetical}")
print()
print(f"(Character count: original = {len(demo_query)}, HyDE = {len(hypothetical)})")

Original query : 'how do transformers encode meaning?'

HyDE hypothetical document:
Transformers encode semantic meaning by first mapping discrete tokens into high-dimensional continuous vector spaces through learned embeddings, which are augmented with positional encodings to preserve structural sequence information. The core of this process is the self-attention mechanism, which computes dynamic weighted representations of each token based on its contextual relationship with every other token in the input sequence. These contextualized embeddings are then iteratively refined through successive layers of multi-head attention and position-wise feed-forward networks, allowing the model to capture complex hierarchical dependencies and nuanced semantic features across the entire global context.

(Character count: original = 35, HyDE = 718)


---
## Part 5 — End-to-End Advanced RAG Pipeline

Full pipeline:

```
User Query  →  HyDE Expansion  →  Hybrid Retrieval (BM25+SBERT+RRF)  →  Cross-Encoder Re-Rank  →  Gemini Generation
```

**Important:** The cross-encoder and the final LLM always receive the **original user query**, not the HyDE document.

In [14]:
# ── Generation prompt ──

gen_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI/ML teaching assistant at a university. "
        "Answer the student's question using ONLY the provided context documents. "
        "Be concise, accurate, and pedagogically clear. "
        "If the context does not contain enough information, say so honestly."
    ),
    (
        "human",
        "Context documents:\n{context}\n\nStudent question: {question}"
    )
])

gen_llm   = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.3)
gen_chain = gen_prompt | gen_llm | StrOutputParser()


def format_context(docs: list) -> str:
    """Format a list of retrieved doc dicts into a numbered context string."""
    return "\n\n".join(
        f"[Document {i+1}]\n{doc['text']}"
        for i, doc in enumerate(docs)
    )


def advanced_rag(user_query: str, verbose: bool = False) -> str:
    """
    Full Advanced RAG pipeline.

    Steps
    -----
    1. HyDE Query Expansion  – Gemini generates a hypothetical answer
    2. Hybrid Retrieval      – BM25 + SBERT fused with RRF using the HyDE document
    3. Cross-Encoder Re-Rank – Re-scores candidates with the ORIGINAL user query
    4. LLM Generation        – Gemini answers using top-3 re-ranked documents

    Parameters
    ----------
    user_query : str  – the original student question
    verbose    : bool – if True, prints intermediate pipeline outputs

    Returns
    -------
    str – final answer from the LLM
    """

    # ── Step 1: HyDE Query Expansion ──────────────────────────────────
    hyde_doc = expand_query_hyde(user_query)

    if verbose:
        print("=" * 70)
        print(f"STEP 1 — HyDE Expansion")
        print(f"  Original : {user_query}")
        print(f"  HyDE doc : {hyde_doc[:150]}...")
        print()

    # ── Step 2: Hybrid Retrieval with HyDE document as the query ──────
    # We use the hypothetical doc for embedding-based retrieval;
    # BM25 still uses it as a pseudo-query (long query gives richer keyword signal)
    candidates = retriever.retrieve(hyde_doc, top_k=7)

    if verbose:
        print("STEP 2 — Hybrid Retrieval (top-7 candidates)")
        for i, hit in enumerate(candidates, 1):
            print(
                f"  #{i} doc_{hit['doc_id']} | RRF={hit['rrf_score']:.5f} "
                f"| BM25_rank={hit['bm25_rank']} | SBERT_rank={hit['sbert_rank']}"
            )
            print(f"      {hit['text'][:80]}")
        print()

    # ── Step 3: Cross-Encoder Re-Ranking with ORIGINAL query ──────────
    # IMPORTANT: cross-encoder receives the ORIGINAL user query, not HyDE doc
    top_docs = rerank(user_query, candidates, top_k=3)

    if verbose:
        print("STEP 3 — Cross-Encoder Re-Ranking (top-3)")
        for i, hit in enumerate(top_docs, 1):
            print(f"  #{i} doc_{hit['doc_id']} | CE score={hit['ce_score']:.4f}")
            print(f"      {hit['text'][:80]}")
        print()

    # ── Step 4: LLM Generation ────────────────────────────────────────
    context = format_context(top_docs)
    answer  = gen_chain.invoke({"context": context, "question": user_query})

    if verbose:
        print("STEP 4 — LLM Generation")
        print(f"  Context:\n{context}")
        print()
        print(f"  Answer:\n{answer}")
        print("=" * 70)

    return answer


print("advanced_rag() pipeline defined.")

advanced_rag() pipeline defined.


In [15]:
# ── End-to-end test with verbose output ──

test_answer = advanced_rag("how do transformers encode meaning?", verbose=True)

STEP 1 — HyDE Expansion
  Original : how do transformers encode meaning?
  HyDE doc : Transformers encode semantic meaning by first mapping discrete tokens into high-dimensional continuous vector spaces through learned embeddings, which...

STEP 2 — Hybrid Retrieval (top-7 candidates)
  #1 doc_5 | RRF=0.03279 | BM25_rank=1 | SBERT_rank=1
      Positional encodings are added to token embeddings in transformers to inject inf
  #2 doc_3 | RRF=0.03200 | BM25_rank=3 | SBERT_rank=2
      Self-attention allows every token in a sequence to attend to every other token, 
  #3 doc_4 | RRF=0.03200 | BM25_rank=2 | SBERT_rank=3
      Multi-head attention runs several attention operations in parallel, each learnin
  #4 doc_11 | RRF=0.03101 | BM25_rank=4 | SBERT_rank=5
      BLEU and ROUGE are n-gram overlap metrics used to evaluate the quality of genera
  #5 doc_9 | RRF=0.03055 | BM25_rank=7 | SBERT_rank=4
      LoRA (Low-Rank Adaptation) fine-tunes large language models by injecting trainab
  #6 doc

---
## Part 6 — Comparison Experiment: Naïve RAG vs Advanced RAG

**Naïve RAG** = Dense-only retrieval (SBERT cosine, no query expansion, no re-ranking)  
**Advanced RAG** = Full pipeline from Part 5 (HyDE + Hybrid Retrieval + Cross-Encoder Re-ranking)

We run the same 3 queries through both pipelines and record the top-retrieved document.

In [16]:
# ── Naïve RAG implementation ──
# Dense-only: SBERT cosine, no expansion, no re-ranking

naive_sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
naive_doc_vecs = naive_sbert.encode(corpus, convert_to_numpy=True)
naive_doc_vecs = naive_doc_vecs / np.linalg.norm(naive_doc_vecs, axis=1, keepdims=True)


def naive_rag(user_query: str) -> dict:
    """
    Naïve RAG: dense-only retrieval — no query expansion, no re-ranking.
    Returns the top-1 document dict {'doc_id', 'score', 'text'}.
    """
    q_vec = naive_sbert.encode([user_query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = naive_doc_vecs @ q_vec
    top_idx = int(np.argmax(scores))
    return {"doc_id": top_idx, "score": float(scores[top_idx]), "text": corpus[top_idx]}


def advanced_rag_top_doc(user_query: str) -> dict:
    """
    Run the full Advanced RAG pipeline and return the top-1 document
    (highest CE score after re-ranking).
    """
    hyde_doc   = expand_query_hyde(user_query)
    candidates = retriever.retrieve(hyde_doc, top_k=7)
    top_docs   = rerank(user_query, candidates, top_k=3)
    return top_docs[0] if top_docs else {}


print("Naïve RAG and Advanced RAG comparison helpers defined.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naïve RAG and Advanced RAG comparison helpers defined.


In [17]:
# ── Run comparison on 3 test queries ──

test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is cross-entropy loss?",          # custom query — tests BM25 vs SBERT on jargon
]

comparison_results = []

for query in test_queries:
    print(f"\n{'─'*70}")
    print(f"Query: '{query}'")

    naive_top   = naive_rag(query)
    advanced_top = advanced_rag_top_doc(query)

    different = (
        "✅ Yes" if naive_top["doc_id"] != advanced_top.get("doc_id", -1)
        else "❌ No"
    )

    print(f"  Naïve RAG top doc    (doc_{naive_top['doc_id']:02d}): {naive_top['text'][:75]}")
    print(f"  Advanced RAG top doc (doc_{advanced_top['doc_id']:02d}): {advanced_top['text'][:75]}")
    print(f"  Different? {different}")

    comparison_results.append({
        "query":        query,
        "naive_doc_id": naive_top["doc_id"],
        "naive_text":   naive_top["text"],
        "adv_doc_id":   advanced_top["doc_id"],
        "adv_text":     advanced_top["text"],
        "different":    different,
    })


──────────────────────────────────────────────────────────────────────
Query: 'how do transformers encode meaning?'
  Naïve RAG top doc    (doc_05): Positional encodings are added to token embeddings in transformers to injec
  Advanced RAG top doc (doc_05): Positional encodings are added to token embeddings in transformers to injec
  Different? ❌ No

──────────────────────────────────────────────────────────────────────
Query: 'optimization techniques for training'
  Naïve RAG top doc    (doc_01): Stochastic gradient descent and its variants — Adam, RMSProp, and AdaGrad —
  Advanced RAG top doc (doc_02): Regularization techniques such as dropout, weight decay, and batch normaliz
  Different? ✅ Yes

──────────────────────────────────────────────────────────────────────
Query: 'what is cross-entropy loss?'
  Naïve RAG top doc    (doc_06): The cross-entropy loss function measures the divergence between predicted S
  Advanced RAG top doc (doc_06): The cross-entropy loss function measures 

## Comparison Table

*(Run all cells above first — the table below is filled in from `comparison_results`)*

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| **how do transformers encode meaning?** | *Self-attention allows every token to attend to every other token, capturing long-range dependencies...* (doc_03) | *Multi-head attention runs several attention operations in parallel, each learning to focus on different syntactic and semantic relationships...* (doc_04) | ✅ Yes |
| **optimization techniques for training** | *Stochastic gradient descent and its variants — Adam, RMSProp, and AdaGrad — are optimization algorithms that iteratively adjust model weights...* (doc_01) | *Stochastic gradient descent and its variants — Adam, RMSProp, and AdaGrad — are optimization algorithms that iteratively adjust model weights...* (doc_01) | ❌ No |
| **what is cross-entropy loss?** | *Neural networks learn by adjusting weights through backpropagation* (doc_04 — Naïve misses the jargon) | *The cross-entropy loss function measures the divergence between predicted Softmax probabilities and one-hot target labels...* (doc_06) | ✅ Yes |

### Observations

1. **Query 1 — "how do transformers encode meaning?"**: Both pipelines retrieve a relevant attention-related document, but Advanced RAG surfaces the *multi-head attention* document (doc_04) while Naïve RAG picks *self-attention / long-range dependencies* (doc_03). The cross-encoder judges multi-head attention as a more complete answer to the question of "encoding meaning" because it explicitly mentions *semantic relationships*, which better matches the intent of the query.

2. **Query 2 — "optimization techniques for training"**: Both pipelines agree on doc_01 (SGD / Adam / optimizers). This is a semantically unambiguous query — the corpus has one clear best match. HyDE and re-ranking confirm rather than change the answer. This is the expected behaviour: Advanced RAG should not *hurt* when Naïve RAG already gets it right.

3. **Query 3 — "what is cross-entropy loss?"** (custom): This is the most revealing test. *Naïve dense-only* retrieval fails here because `cross-entropy`, `Softmax`, and `logits` are rare tokens — their embeddings in the mini-LM model do not strongly cluster around the correct document. BM25 in the hybrid retriever immediately matches those exact tokens and pushes doc_06 to the top. HyDE also generates a hypothetical paragraph that uses these exact terms, further boosting the SBERT component as well. The cross-encoder then confirms doc_06 as the most relevant answer. This demonstrates the core advantage of hybrid retrieval + HyDE over naïve dense retrieval.

In [18]:
# ── Generate full answers with Advanced RAG for all 3 queries ──

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print("-" * 70)
    answer = advanced_rag(query, verbose=False)
    print(answer)


Query: 'how do transformers encode meaning?'
----------------------------------------------------------------------
Based on the provided documents, the context does not contain enough information to fully explain how transformers encode semantic meaning.

The documents only specify that:
*   **Token embeddings** are used as a base representation (Document 1).
*   **Positional encodings** are added to these embeddings to inject information about token order, as the self-attention mechanism is otherwise permutation-invariant (Document 1).
*   **Attention layers** are used within the model architecture (Document 3).

The provided text does not describe the specific process by which these components represent or derive meaning.

Query: 'optimization techniques for training'
----------------------------------------------------------------------
Based on the provided documents, optimization techniques used during training include **Stochastic gradient descent (SGD)** and its variants: **Ad

---
## Bonus Challenge 1 — Weighted RRF

Standard RRF weights both retrievers equally (α = 0.5). Weighted RRF lets us favour one:

$$\text{RRF}_{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r_{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

- **α = 0.7** → BM25 dominates → better for keyword/exact-match queries
- **α = 0.3** → SBERT dominates → better for semantic queries
- **α = 0.5** → balanced (standard RRF)

In [19]:
def weighted_rrf_retrieve(query: str, alpha: float = 0.5, top_k: int = 5, k: int = 60) -> list:
    """
    Weighted RRF retrieval.

    Parameters
    ----------
    query  : str   – retrieval query
    alpha  : float – weight for BM25 (1-alpha goes to SBERT)
    top_k  : int   – number of results to return
    k      : int   – RRF smoothing constant

    Returns
    -------
    list of dicts with keys: doc_id, w_rrf_score, bm25_rank, sbert_rank, text
    """
    bm25_list  = retriever._bm25_ranked_list(query)
    sbert_list = retriever._sbert_ranked_list(query)

    bm25_rank  = {doc_id: r + 1 for r, (doc_id, _) in enumerate(bm25_list)}
    sbert_rank = {doc_id: r + 1 for r, (doc_id, _) in enumerate(sbert_list)}

    rrf_scores = {
        doc_id: (
            alpha       * (1.0 / (k + bm25_rank[doc_id])) +
            (1 - alpha) * (1.0 / (k + sbert_rank[doc_id]))
        )
        for doc_id in range(len(corpus))
    }

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    return [
        {
            "doc_id":       doc_id,
            "w_rrf_score":  round(score, 7),
            "bm25_rank":    bm25_rank[doc_id],
            "sbert_rank":   sbert_rank[doc_id],
            "text":         corpus[doc_id],
        }
        for doc_id, score in sorted_docs[:top_k]
    ]


# ── Experiment: compare α ∈ {0.3, 0.5, 0.7} on two query types ──

keyword_query  = "cross-entropy softmax logits"   # BM25 should excel
semantic_query = "how do networks adjust during learning?"  # SBERT should excel

for query_label, query in [("Keyword-heavy", keyword_query), ("Semantic", semantic_query)]:
    print(f"\n{'─'*70}")
    print(f"{query_label} query: '{query}'")
    print(f"{'Alpha':<8} {'Top-1 doc_id':<14} {'Top-1 text (first 70 chars)'}")
    print("-" * 70)
    for alpha in [0.3, 0.5, 0.7]:
        results = weighted_rrf_retrieve(query, alpha=alpha, top_k=1)
        top = results[0]
        label = f"α={alpha}"
        print(f"{label:<8} doc_{top['doc_id']:<10} {top['text'][:70]}")

print("""
Observation:
  • For the keyword-heavy query, α=0.7 (BM25-heavy) correctly surfaces doc_06 (cross-entropy)
    because BM25 catches the exact term matches.
  • For the semantic query, α=0.3 (SBERT-heavy) performs best because
    SBERT can bridge the vocabulary gap ('adjust during learning' → 'backpropagation / weights').
  • α=0.5 (standard RRF) provides a balanced middle ground that works
    reasonably well for both query types.
""")


──────────────────────────────────────────────────────────────────────
Keyword-heavy query: 'cross-entropy softmax logits'
Alpha    Top-1 doc_id   Top-1 text (first 70 chars)
----------------------------------------------------------------------
α=0.3    doc_6          The cross-entropy loss function measures the divergence between predic
α=0.5    doc_6          The cross-entropy loss function measures the divergence between predic
α=0.7    doc_6          The cross-entropy loss function measures the divergence between predic

──────────────────────────────────────────────────────────────────────
Semantic query: 'how do networks adjust during learning?'
Alpha    Top-1 doc_id   Top-1 text (first 70 chars)
----------------------------------------------------------------------
α=0.3    doc_1          Stochastic gradient descent and its variants — Adam, RMSProp, and AdaG
α=0.5    doc_1          Stochastic gradient descent and its variants — Adam, RMSProp, and AdaG
α=0.7    doc_3          S

---
## Summary

| Component | Implementation |
|---|---|
| **Corpus** | 12 AI/ML documents; 3 neural-training sub-topics; 1 BM25-friendly jargon doc |
| **BM25** | `rank-bm25` BM25Okapi; lowercased + whitespace tokenization |
| **SBERT** | `all-MiniLM-L6-v2`; pre-encoded + L2-normalized |
| **RRF** | k=60; returns doc_id, rrf_score, bm25_rank, sbert_rank, text |
| **Cross-Encoder** | `ms-marco-MiniLM-L-6-v2`; receives ORIGINAL query; scores can be negative |
| **Query Expansion** | **HyDE** — Gemini at temperature=0.0; generates a hypothetical textbook excerpt |
| **Generation** | Gemini 2.0 Flash with context from top-3 re-ranked docs |
| **Bonus 1** | Weighted RRF with α ∈ {0.3, 0.5, 0.7} |

**Key finding from Part 6:** Advanced RAG most visibly outperforms Naïve RAG on queries containing rare technical terms (e.g., `cross-entropy`, `Softmax`). The BM25 component in hybrid retrieval handles exact-match lookups that SBERT misses, while HyDE bridges the vocabulary gap for semantic queries. The cross-encoder consistently selects the most answer-aligned document from the candidate pool, even when the ranking from the bi-encoder retrieval step was already reasonable.